# 01 · Processing gain

**Question:** why can a signal that is *weaker than the noise* still stand clearly above the floor in a spectrum?

**Idea in three lines**
- Noise is spread over the whole span (fs). An FFT bin only "sees" the noise inside its own resolution bandwidth (RBW ≈ fs / nfft × window factor).
- A tone sits in a single bin, so the bin holds the tone's full power.
- So the tone-to-floor ratio in the spectrum = SNR in the whole span + **processing gain**, where processing gain = 10·log10(fs / RBW). Doubling nfft adds ~3 dB.

Averaging (Welch) is a different thing: it does **not** lower the floor, it makes it **smoother**, so weak signals are easier to spot.

Change the parameters in the next cell and re-run everything (*Run All*).

In [ ]:
# ---- Parameters: change these ----
fs = 2e6               # sample rate, Hz
noise_dbfs = -30.0     # total noise power over the whole span, dBFS
tone_dbfs = -50.0      # tone power, dBFS (here 20 dB BELOW the noise)
tone_offset_hz = 300e3 # tone frequency relative to center
nfft_list = [256, 1024, 4096, 16384]
duration_s = 0.5
seed = 0

In [ ]:
import numpy as np
import plotly.graph_objects as go
from tayf.dsp.spectrum import welch_power, rbw_hz

rng = np.random.default_rng(seed)
n = int(fs * duration_s)
t = np.arange(n) / fs
noise = np.sqrt(10**(noise_dbfs/10) / 2) * (rng.standard_normal(n) + 1j*rng.standard_normal(n))
tone = 10**(tone_dbfs/20) * np.exp(2j*np.pi*tone_offset_hz*t)
x = (noise + tone).astype(np.complex64)
print(f"SNR over the whole span: {tone_dbfs - noise_dbfs:+.1f} dB")

## 1 · In the time domain the tone is invisible
The real part (I) of the first 400 samples, with and without the tone. They look the same.

In [ ]:
fig = go.Figure()
fig.add_scatter(y=noise[:400].real, name="noise only", line=dict(width=1))
fig.add_scatter(y=x[:400].real, name="noise + tone", line=dict(width=1))
fig.update_layout(height=300, xaxis_title="sample", yaxis_title="I (full scale = 1)", margin=dict(t=30))
fig.show()

## 2 · In the spectrum it appears, and grows with nfft
Each curve is a Welch spectrum with a different FFT size. Watch the floor drop while the tone stays at its true power.

In [ ]:
fig = go.Figure()
rows = []
for nfft in nfft_list:
    f, p, k = welch_power(x, fs, nfft=nfft)
    fig.add_scatter(x=f/1e3, y=p, name=f"nfft={nfft}", line=dict(width=1))
    rbw = rbw_hz(fs, nfft)
    floor = np.median(p)
    rows.append((nfft, rbw, 10*np.log10(fs/rbw), noise_dbfs + 10*np.log10(rbw/fs), floor, p.max(), p.max() - floor, k))
fig.update_layout(height=380, xaxis_title="frequency offset (kHz)", yaxis_title="dBFS / bin", margin=dict(t=30))
fig.show()

print(f"{'nfft':>6} {'RBW Hz':>9} {'proc gain dB':>12} {'floor pred':>10} {'floor meas':>10} {'tone dBFS':>9} {'tone-floor':>10} {'averages':>8}")
for r in rows:
    print(f"{r[0]:>6} {r[1]:>9.1f} {r[2]:>12.1f} {r[3]:>10.1f} {r[4]:>10.1f} {r[5]:>9.1f} {r[6]:>10.1f} {r[7]:>8}")

**Read the table:**
- *floor pred* = noise_dbfs + 10·log10(RBW/fs). The measured floor (median) matches it when many segments are averaged. With few averages (large nfft, short data) it reads up to 1.6 dB low: the median of a single FFT's noise power is below its mean (ln 2).
- *tone − floor* ≈ (span SNR) + (processing gain): it rises ~6 dB each time nfft is ×4.
- The tone reads close to its true power (it can read up to 1.4 dB low when it falls between two bins: Hann *scalloping loss*).

## 3 · Averaging smooths the floor, it does not lower it
Same nfft, one FFT versus many averaged. The mean level is the same; the fluctuation shrinks (by ~1/√K).

In [ ]:
nfft = 2048
f1, p1, k1 = welch_power(x[:nfft], fs, nfft=nfft)          # a single FFT
fk, pk, kk = welch_power(x, fs, nfft=nfft)                 # many averaged
fig = go.Figure()
fig.add_scatter(x=f1/1e3, y=p1, name=f"K = {k1}", line=dict(width=1))
fig.add_scatter(x=fk/1e3, y=pk, name=f"K = {kk}", line=dict(width=1))
fig.update_layout(height=350, xaxis_title="frequency offset (kHz)", yaxis_title="dBFS / bin", margin=dict(t=30))
fig.show()
mask = np.abs(f1 - tone_offset_hz) > 20e3
print(f"floor std: K={k1}: {p1[mask].std():.2f} dB   K={kk}: {pk[mask].std():.2f} dB")
print(f"floor mean (linear, in dB): K={k1}: {10*np.log10(np.mean(10**(p1[mask]/10))):.1f}   K={kk}: {10*np.log10(np.mean(10**(pk[mask]/10))):.1f}")

## Try it yourself
- Set `tone_dbfs = -70` (40 dB below the noise). Which nfft do you need to see it?
- A wideband signal (like FM, ~200 kHz) does **not** get this gain: its power spreads over many bins, just like noise. Processing gain helps narrow signals.
- This is why the old detector tests saw a "weak" tone easily: at nfft = 2048 the processing gain is about 31 dB with a Hann window (33 dB = 10·log10(2048) without the window factor).

The dashboard's readout shows the floor both per bin (dBFS/bin) and normalised (dBFS/Hz). Changing the FFT size moves the first, not the second.